# Funciones para visualizar resultados

In [ ]:
import os
import re
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np

# ──────────────────────────────────────────────────────────────────────────────
# CONFIGURACIÓN DE MÉTRICAS POR TAREA
# ──────────────────────────────────────────────────────────────────────────────

METRICAS_MOLECULAR_SUBTYPE = {
    "Clasificación": [
        "accuracy", "kappa",
        "f1_macro", "f1_micro", "f1_weighted",
        "precision_macro", "precision_micro", "precision_weighted",
        "recall_macro", "recall_micro", "recall_weighted",
    ],
    "Ranking / AUC": [
        "auroc_macro", "auroc_micro", "auroc_weighted",
        "auprc_macro", "auprc_micro", "auprc_weighted",
    ],
}

METRICAS_BINARIA = {
    "Clasificación": [
        "accuracy", "kappa",
        "f1_macro", "f1_weighted",
        "precision_macro", "precision_weighted",
        "recall_macro", "recall_weighted",
    ],
    "Ranking / AUC": [
        "auroc", "auprc",
    ],
}

TASK_LABEL = {
    "molecular_subtype": "Subtipo molecular (4 clases)",
    "er":   "ER (binario)",
    "pr":   "PR (binario)",
    "her2": "HER2 (binario)",
    "ki67": "Ki67 (binario)",
}

# ──────────────────────────────────────────────────────────────────────────────
# HELPERS
# ──────────────────────────────────────────────────────────────────────────────

def detectar_task_desde_ruta(ruta_carpeta):
    """
    Extrae la task del nombre de la carpeta buscando 'task-<nombre>'.
    Ejemplo: ...task-er_weighted... → 'er'
             ...task-molecular_subtype_weighted... → 'molecular_subtype'
    """
    nombre = os.path.basename(os.path.normpath(ruta_carpeta))
    # Intentar primero molecular_subtype (tiene guion bajo en el nombre)
    if "task-molecular_subtype" in nombre:
        return "molecular_subtype"
    match = re.search(r'task-([a-zA-Z0-9]+?)(?:_|$)', nombre)
    if match:
        return match.group(1)
    return None


def grupo_metricas(task):
    if task == "molecular_subtype":
        return METRICAS_MOLECULAR_SUBTYPE
    return METRICAS_BINARIA


def nombre_limpio(col):
    return col.replace("_", " ").title()


# ──────────────────────────────────────────────────────────────────────────────
# PARTE 1: GRÁFICAS EVOLUTIVAS TRAIN VS VAL
# ──────────────────────────────────────────────────────────────────────────────

def plot_evolucion(df_train, df_val, task, ruta_carpeta):
    grupos   = grupo_metricas(task)
    task_str = TASK_LABEL.get(task, task)

    for nombre_grupo, cols_grupo in grupos.items():
        cols = [c for c in cols_grupo if c in df_train.columns and c in df_val.columns]
        if not cols:
            continue

        n_cols = 3
        n_rows = int(np.ceil(len(cols) / n_cols))

        fig, axes = plt.subplots(n_rows, n_cols, figsize=(16, 4 * n_rows))
        fig.suptitle(
            f"Evolución por época — {nombre_grupo}\nTask: {task_str}",
            fontsize=14, fontweight="bold", y=1.01
        )

        axes_flat = np.array(axes).flatten()

        for i, col in enumerate(cols):
            ax = axes_flat[i]

            ax.plot(df_train["epoch"], df_train[col],
                    label="Train", color="#2563EB", linewidth=1.8,
                    marker=".", markersize=4, linestyle="-")
            ax.plot(df_val["epoch"], df_val[col],
                    label="Val", color="#DC2626", linewidth=1.8,
                    marker=".", markersize=4, linestyle="--")

            ax.set_title(nombre_limpio(col), fontsize=11)
            ax.set_xlabel("Época", fontsize=9)
            ax.set_ylabel("Valor", fontsize=9)
            ax.legend(fontsize=9)
            ax.grid(True, alpha=0.25)
            ax.set_ylim(bottom=0)

            # Marcar mejor val con línea vertical y anotación
            idx_best   = df_val[col].idxmax()
            best_epoch = df_val.loc[idx_best, "epoch"]
            best_val   = df_val.loc[idx_best, col]
            ax.axvline(best_epoch, color="#DC2626", linewidth=0.8,
                       linestyle=":", alpha=0.6)
            ax.annotate(f"{best_val:.3f}",
                        xy=(best_epoch, best_val),
                        xytext=(4, -12), textcoords="offset points",
                        fontsize=8, color="#DC2626")

        for j in range(len(cols), len(axes_flat)):
            axes_flat[j].axis("off")

        plt.tight_layout()
        slug     = nombre_grupo.lower().replace(" ", "_").replace("/", "")
        ruta_fig = os.path.join(ruta_carpeta, f"evolucion_{task}_{slug}.png")
        plt.savefig(ruta_fig, dpi=120, bbox_inches="tight")
        print(f"  Guardada: {ruta_fig}")
        plt.show()


# ──────────────────────────────────────────────────────────────────────────────
# PARTE 2: TABLA DE TEST
# ──────────────────────────────────────────────────────────────────────────────

def mostrar_tabla_test(ruta_carpeta, task):
    ruta = os.path.join(ruta_carpeta, "test_metrics.csv")
    if not os.path.exists(ruta):
        print(f"\n  No se encontró test_metrics.csv en {ruta_carpeta}")
        return

    try:
        df     = pd.read_csv(ruta)
        fila   = df.iloc[-1]
        grupos = grupo_metricas(task)

        print(f"\n{'═'*65}")
        print(f"  RESULTADOS TEST")
        print(f"  Task: {TASK_LABEL.get(task, task)}")
        print(f"{'═'*65}")

        for nombre_grupo, cols_grupo in grupos.items():
            cols = [c for c in cols_grupo if c in fila.index]
            if not cols:
                continue

            print(f"\n  {nombre_grupo}")
            print(f"  {'Métrica':<38} │ {'Valor':>10}")
            print(f"  {'─'*38}─┼─{'─'*10}")

            for col in cols:
                valor   = fila[col]
                val_str = f"{valor:.4f}" if isinstance(valor, (float, int)) else str(valor)
                print(f"  {nombre_limpio(col):<38} │ {val_str:>10}")

        print(f"\n{'═'*65}\n")

    except Exception as e:
        print(f"  Error leyendo test_metrics.csv: {e}")


# ──────────────────────────────────────────────────────────────────────────────
# ENTRADA PRINCIPAL
# ──────────────────────────────────────────────────────────────────────────────

def analizar_experimento(ruta_carpeta):
    if not os.path.exists(ruta_carpeta):
        print(f"La ruta '{ruta_carpeta}' no existe.")
        return

    task = detectar_task_desde_ruta(ruta_carpeta)
    if task is None:
        print("No se pudo detectar la task desde el nombre de la carpeta.")
        print("Asegúrate de que el nombre contiene 'task-<nombre>'.")
        return

    print(f"\nTask detectada : {TASK_LABEL.get(task, task)}")
    print(f"Carpeta        : {os.path.basename(os.path.normpath(ruta_carpeta))}\n")

    # ── Gráficas evolutivas ───────────────────────────────────────────────────
    ruta_train = os.path.join(ruta_carpeta, "train_metrics.csv")
    ruta_val   = os.path.join(ruta_carpeta, "val_metrics.csv")

    if os.path.exists(ruta_train) and os.path.exists(ruta_val):
        df_train = pd.read_csv(ruta_train).sort_values("epoch")
        df_val   = pd.read_csv(ruta_val).sort_values("epoch")
        print(f"Épocas registradas: {len(df_train)}")
        print("Generando gráficas...\n")
        plot_evolucion(df_train, df_val, task, ruta_carpeta)
    else:
        print("No se encontraron train_metrics.csv o val_metrics.csv.")

    # ── Tabla de test ─────────────────────────────────────────────────────────
    mostrar_tabla_test(ruta_carpeta, task)


# ──────────────────────────────────────────────────────────────────────────────
# EJECUCIÓN
# ──────────────────────────────────────────────────────────────────────────────

ruta = r"/workspace/results/FULL_DGX_20260317_173430_task-er_weighted_loss_model-VIT_unfrozen_MIL-attentionMIL_AdamW_lr1e-05_wd0.001_bs32_ep100"
analizar_experimento(ruta)
